In [2]:
# /workspace/notebooks/gt_brs_score.ipynb
import os
import pandas as pd

from modules.config import *
from modules.utils import *
from modules.metrics import *
from modules.experiment_set import select_set

In [3]:
experiment_set1 = select_set("set1")

In [4]:
ground_truth_set1 = get_ground_truth(replays=list(experiment_set1.test_replays))

In [5]:
inference_vanilla_set1 = get_inference(experiment_set1.model_vanilla(),
                                       replays=list(experiment_set1.test_replays),
                                       k=5)

In [6]:
inference_kbrs_set1 = get_inference(experiment_set1.model_kbrs(), 
                                    replays=list(experiment_set1.test_replays),
                                    k=5)

In [7]:
from enum import Enum

class Channel(Enum):
    Player_1_Worker = 0
    Player_1_Ground = 1
    Player_1_Air = 2
    Player_1_Building = 3
    
    Player_2_Worker = 4
    Player_2_Ground = 5
    Player_2_Air = 6
    Player_2_Building = 7
    
    Resource = 8
    Vision = 9
    Terrain = 10

In [8]:
def get_item_npy_obj(item):
    npy_path = os.path.join(DATA_DIR,
                            'input',
                            'dst',
                            item['replay'].item(),
                            str(item['image_id'].item())
                            + '.npy')
    obj = np.load(npy_path)
    return obj

In [9]:
def get_xywhs(item_row, columns: list):
    gts = []
    for k in GT_COLS:
        if k in item_row:
            v = item_row[k]
            if v is None: 
                continue
            arr = np.asarray(v, dtype=int).ravel()
            if arr.size == 4 and not np.any(np.isnan(arr)):
                x, y, w, h = arr
                gts.append((x,y,w,h))
    return gts


In [10]:
GT_COLS = [('gt', f'{i}') for i in range(5)]
VANILLA_COLS = [('vanilla', f'{i}') for i in range(5)]
KBRS_COLS = [('kbrs', f'{i}') for i in range(5)]

In [11]:
def get_kernel_density_score(arr, xywh, kw=20, kh=12, use_bbox=False) -> float:
    x_sum = arr.sum(axis=0, keepdims=False).astype(np.float32, copy=False)  # (H,W)

    H, W = x_sum.shape

    x, y, w, h = [_ for _ in xywh]
    if use_bbox:
        k_w = max(1, int(round(w)))
        k_h = max(1, int(round(h)))
    else:
        k_w = int(kw)
        k_h = int(kh)

    if k_w > W or k_h > H:
        raise ValueError(f"kernel size ({k_w},{k_h}) exceeds array size ({W},{H})")

    # 적분영상 (상단/좌측 0 padding)
    ii = np.pad(x_sum, ((1, 0), (1, 0)), mode='constant')
    ii = ii.cumsum(axis=0).cumsum(axis=1)

    # 모든 위치의 윈도 합 (H-k_h+1, W-k_w+1)
    win_sum = (
        ii[k_h:, k_w:]      # br
        - ii[:-k_h, k_w:]   # tr
        - ii[k_h:, :-k_w]   # bl
        + ii[:-k_h, :-k_w]  # tl
    )
    den_map = win_sum / float(k_h * k_w)  # 평균값

    # 정규화 범위
    dmin = float(den_map.min())
    dmax = float(den_map.max())
    if dmax == dmin:
        # 모든 값이 동일하면 점수 0으로 정의
        norm_map = np.zeros_like(den_map, dtype=np.float32)
    else:
        norm_map = (den_map - dmin) / (dmax - dmin)

    # 좌표 보정(라운드 + 경계 클리핑)
    xi = int(round(x))
    yi = int(round(y))
    xi = max(0, min(W - k_w, xi))
    yi = max(0, min(H - k_h, yi))

    # norm_map의 인덱스는 (top-left y, x)
    return float(norm_map[yi, xi])    

In [12]:
def _resolve_kernel_size(xywh, kw, kh, use_bbox):
    x, y, w, h = xywh
    if use_bbox:
        k_w = max(1, int(round(w)))
        k_h = max(1, int(round(h)))
    else:
        k_w = int(kw)
        k_h = int(kh)
    return int(round(x)), int(round(y)), k_w, k_h

def _clip_top_left(xi, yi, W, H, k_w, k_h):
    xi = max(0, min(W - k_w, xi))
    yi = max(0, min(H - k_h, yi))
    return xi, yi

In [13]:
def get_kernel_centeredness_score(arr: np.ndarray,
                                  xywh,
                                  kw: int = 20,
                                  kh: int = 12,
                                  sigma=None,
                                  use_bbox: bool = False) -> float:
    x = arr

    _, H, W = x.shape
    xi, yi, k_w, k_h = _resolve_kernel_size(xywh, kw, kh, use_bbox)
    if k_w > W or k_h > H:
        raise ValueError(f"kernel ({k_w},{k_h}) exceeds array size ({W},{H})")

    # 커널 맵 계산(정규화 포함)
    cen_map = score_centeredness(x, stride=1, sigma=sigma, kw=k_w, kh=k_h)  # (H-k_h+1, W-k_w+1)

    # 좌표 클리핑 후 스칼라 추출
    xi, yi = _clip_top_left(xi, yi, W, H, k_w, k_h)
    val = float(cen_map[yi, xi])

    # NaN/inf 보호
    if not np.isfinite(val):
        val = 0.0
    return val

In [14]:
def get_kernel_mixture_score(arr: np.ndarray,
                             xywh,
                             kw: int = 20,
                             kh: int = 12,
                             use_bbox: bool = False,
                             mode: str = 'entropy',
                             fill_when_const: float = 0.0,
                             eps: float = 1e-6,
                             pow_k: float = 1.0) -> float:
    x = arr

    _, H, W = x.shape
    xi, yi, k_w, k_h = _resolve_kernel_size(xywh, kw, kh, use_bbox)
    if k_w > W or k_h > H:
        raise ValueError(f"kernel ({k_w},{k_h}) exceeds array size ({W},{H})")

    # 커널 맵 계산(정규화 포함)
    mix_map = score_mixture(
        x, stride=1, mode=mode,
        fill_when_const=fill_when_const, eps=eps, pow_k=pow_k,
        kw=k_w, kh=k_h
    )  # (H-k_h+1, W-k_w+1)

    # 좌표 클리핑 후 스칼라 추출
    xi, yi = _clip_top_left(xi, yi, W, H, k_w, k_h)
    val = float(mix_map[yi, xi])

    if not np.isfinite(val):
        val = 0.0
    return val

In [15]:
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

META_TOPS = {"replay", "image_id"}

def ensure_3level_xywh_inplace(df: pd.DataFrame) -> pd.DataFrame:
    """
    (group,id) -> (group,id,metric) in-place 승격
      - group ∈ META_TOPS: metric='value'
      - 그 외:               metric='xywh'
    """
    if df.columns.nlevels == 3:
        return df
    if df.columns.nlevels != 2:
        raise ValueError("merged.columns must have 2 or 3 levels.")
    new_cols = []
    for g, i in df.columns:
        metric = "value" if g in META_TOPS else "xywh"
        new_cols.append((g, i, metric))
    df.columns = pd.MultiIndex.from_tuples(new_cols, names=["group","id","metric"])
    return df

In [16]:
def _as_xywh(v):
    if v is None:
        return (np.nan, np.nan, np.nan, np.nan)
    if isinstance(v, dict):
        for keys in (("x","y","w","h"), ("X","Y","W","H")):
            if all(k in v for k in keys):
                return (float(v[keys[0]]), float(v[keys[1]]),
                        float(v[keys[2]]), float(v[keys[3]]))
        vals = list(v.values())
        if len(vals) >= 4:
            a = np.asarray(vals, dtype=float).ravel()
            return tuple(a[:4])
        return (np.nan, np.nan, np.nan, np.nan)
    try:
        a = np.asarray(v, dtype=float).ravel()
        if a.size >= 4:
            return (float(a[0]), float(a[1]), float(a[2]), float(a[3]))
    except Exception:
        pass
    return (np.nan, np.nan, np.nan, np.nan)

# ------- 점수 맵 샘플링 유틸 -------
def _clip_xy(xi, yi, W, H, kw, kh):
    xi = max(0, min(W - kw, int(round(xi))))
    yi = max(0, min(H - kh, int(round(yi))))
    return xi, yi

def _collect_groups_ids(df: pd.DataFrame):
    tops = [g for g in df.columns.get_level_values(0).unique() if g not in META_TOPS]
    gid_map = {g: sorted({i for (g2,i,_) in df.columns if g2==g}) for g in tops}
    return gid_map  # {'gt':['0','1',...], 'kbrs':[...], ...}

def _ensure_metric_columns(df: pd.DataFrame, gid_map, metrics=("density","centeredness","mixture")):
    for g, ids in gid_map.items():
        for i in ids:
            for m in metrics:
                col = (g, i, m)
                if col not in df.columns:
                    df[col] = np.nan

In [17]:
# # ------- 메인: 모든 행 배치 처리 -------
# def score_all_rows(
#     merged: pd.DataFrame,
#     arr_getter,                    # callable(replay, image_id) -> np.ndarray (C,H,W)
#     kw: int = 20, kh: int = 12,
#     use_bbox: bool = False,        # True면 각 xywh의 (w,h) 사용
#     sigma=None,                    # centeredness용
#     mode: str = "entropy",         # mixture용
#     fill_when_const: float = 0.0, eps: float = 1e-6, pow_k: float = 1.0,
#     parallel_workers: int = 0      # 0이면 단일 스레드, >0이면 ThreadPool 병렬
# ):
#     """
#     merged의 모든 행에 대해:
#       1) arr_getter로 (C,H,W) 로드 + channel_slice 적용
#       2) 고정커널이면 맵 한 번, use_bbox=True면 (w,h) 고유값 그룹별 맵 계산
#       3) 각 (group,id) xywh에서 (den/cen/mix) 샘플해 (group,id,metric)에 기록
#     """
#     df = ensure_3level_xywh_inplace(merged)
#     gid_map = _collect_groups_ids(df)
#     _ensure_metric_columns(df, gid_map)
    
#     rcol = ("replay","", "value")
#     icol = ("image_id","", "value")
    
#     if rcol not in df.columns or icol not in df.columns:
#         raise ValueError("merged에 ('replay','', 'value'), ('image_id','', 'value')가 필요합니다.")

#     # ---- 행 단위 작업 함수 ----
#     def _process_row(idx):
#         rep = df.at[idx, rcol]
#         img = df.at[idx, icol]
#         if pd.isna(rep) or pd.isna(img):
#             return idx, "skip_meta"

#         arr = arr_getter(rep, img)           # (C,H,W)
#         if arr is None:
#             return idx, "no_arr"

#         x = arr
#         _, H, W = x.shape

#         # 이 행에서 필요한 모든 xywh 수집
#         xy_map = {}  # (g,i) -> (x,y,w,h)
#         ks_set = set()
#         for g, ids in gid_map.items():
#             for i in ids:
#                 v = df.at[idx, (g, i, "xywh")] if (g, i, "xywh") in df.columns else np.nan
#                 xy = _as_xywh(v)
#                 xy_map[(g,i)] = xy
#                 if use_bbox:
#                     w = max(1, int(round(xy[2]))) if np.isfinite(xy[2]) else kw
#                     h = max(1, int(round(xy[3]))) if np.isfinite(xy[3]) else kh
#                     ks_set.add((min(w, W), min(h, H)))
#         if not use_bbox:
#             ks_set.add((kw, kh))

#         # (w,h) 그룹별로 맵 계산
#         cache = {}  # (kw,kh) -> (den_map, cen_map, mix_map)
#         results_local = {}  # (g,i) -> (d,c,m)

#         for k_w, k_h in ks_set:
#             # 맵 한 번 계산
#             den_map = score_density(x, kw=k_w, kh=k_h)  # (H-k_h+1, W-k_w+1)
#             cen_map = score_centeredness(x, kw=k_w, kh=k_h, sigma=sigma)
#             mix_map = score_mixture(x, kw=k_w, kh=k_h, mode=mode,
#                                     fill_when_const=fill_when_const, eps=eps, pow_k=pow_k)
#             cache[(k_w,k_h)] = (den_map, cen_map, mix_map)

#         # 샘플링해서 결과 기록
#         for (g,i), (xx,yy,ww,hh) in xy_map.items():
#             k_w, k_h = (kw, kh) if not use_bbox else (
#                 max(1, int(round(ww))) if np.isfinite(ww) else kw,
#                 max(1, int(round(hh))) if np.isfinite(hh) else kh)
#             k_w, k_h = min(k_w, W), min(k_h, H)

#             den_map, cen_map, mix_map = cache[(k_w,k_h)]
#             xi, yi = _clip_xy(xx, yy, W, H, k_w, k_h)

#             d = float(den_map[yi, xi])
#             c = float(cen_map[yi, xi])
#             m = float(mix_map[yi, xi])
#             results_local[(g,i)] = (d,c,m)

#         # DataFrame에 일괄 반영
#         for (g,i), (d,c,m) in results_local.items():
#             df.at[idx, (g,i,"density")] = d
#             df.at[idx, (g,i,"centeredness")] = c
#             df.at[idx, (g,i,"mixture")] = m

#         return idx, "ok"

#     # ---- 실행 (단일/병렬) ----
#     indices = list(df.index)
#     if parallel_workers and parallel_workers > 0:
#         with ThreadPoolExecutor(max_workers=parallel_workers) as ex:
#             futs = {ex.submit(_process_row, idx): idx for idx in indices}
#             for _ in as_completed(futs):
#                 pass
#     else:
#         for idx in indices:
#             _process_row(idx)

In [19]:
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

In [20]:
# from functools import lru_cache

# def _safe_minmax_norm(a: np.ndarray, fill=0.0) -> np.ndarray:
#     if a.size == 0:
#         return np.empty_like(a, dtype=np.float32)
#     mn = np.nanmin(a); mx = np.nanmax(a); rng = mx - mn
#     if not np.isfinite(rng) or rng == 0:
#         return np.full_like(a, fill, dtype=np.float32)
#     out = (a - mn) / rng
#     return out.astype(np.float32, copy=False)

# def _integral_image(x2d: np.ndarray) -> np.ndarray:
#     # pad(1,1) 후 누적합
#     ii = np.pad(x2d, ((1,0),(1,0)), mode='constant')
#     ii = ii.cumsum(axis=0).cumsum(axis=1)
#     return ii

# def _box_sum(ii: np.ndarray, kw: int, kh: int) -> np.ndarray:
#     # 적분영상에서 윈도 합: 'valid' 영역 (H-kh+1, W-kw+1)
#     return ii[kh:, kw:] - ii[:-kh, kw:] - ii[kh:, :-kw] + ii[:-kh, :-kw]

# @lru_cache(maxsize=128)
# def _gaussian_kernel2d_cached(kw: int, kh: int, sigma):
#     # sigma=None이면 대략적인 기본값 (kw/6, kh/6) 정도를 사용
#     if sigma is None:
#         sx = max(1e-6, kw / 6.0)
#         sy = max(1e-6, kh / 6.0)
#     elif np.isscalar(sigma):
#         sx = sy = float(sigma)
#     else:
#         sx, sy = float(sigma[0]), float(sigma[1])

#     x = np.arange(kw) - (kw - 1) / 2.0
#     y = np.arange(kh) - (kh - 1) / 2.0
#     X, Y = np.meshgrid(x, y)  # (kh,kw)
#     K = np.exp(-0.5 * ((X / (sx + 1e-6))**2 + (Y / (sy + 1e-6))**2)).astype(np.float32, copy=False)
#     s = K.sum()
#     if s <= 0:
#         K[:] = 0; s = 1.0
#     return (K / s).astype(np.float32, copy=False)

# def _centeredness_map(x_sum: np.ndarray, kw: int, kh: int, sigma=None) -> np.ndarray:
#     # 슬라이딩 뷰 + 커널 내적 (작은 커널은 이게 제일 빠릅니다)
#     from numpy.lib.stride_tricks import sliding_window_view
#     K = _gaussian_kernel2d_cached(kw, kh, None if sigma is None else tuple(np.atleast_1d(sigma)))
#     patches = sliding_window_view(x_sum, (kh, kw))  # (H-kh+1, W-kw+1, kh, kw)
#     cen = (patches * K).sum(axis=(-1, -2))          # (H-kh+1, W-kw+1)
#     return _safe_minmax_norm(cen, fill=0.0)

# def _as_xywh(v):
#     if v is None: return (np.nan, np.nan, np.nan, np.nan)
#     if isinstance(v, dict):
#         for keys in (("x","y","w","h"), ("X","Y","W","H")):
#             if all(k in v for k in keys):
#                 return (float(v[keys[0]]), float(v[keys[1]]),
#                         float(v[keys[2]]), float(v[keys[3]]))
#         vals = list(v.values())
#         if len(vals) >= 4:
#             a = np.asarray(vals, dtype=float).ravel()
#             return tuple(a[:4])
#         return (np.nan, np.nan, np.nan, np.nan)
#     try:
#         a = np.asarray(v, dtype=float).ravel()
#         if a.size >= 4:
#             return (float(a[0]), float(a[1]), float(a[2]), float(a[3]))
#     except Exception:
#         pass
#     return (np.nan, np.nan, np.nan, np.nan)

# def _clip_xy(xi, yi, W, H, kw, kh):
#     if not np.isfinite(xi): xi = 0
#     if not np.isfinite(yi): yi = 0
#     xi = int(round(xi)); yi = int(round(yi))
#     return max(0, min(W - kw, xi)), max(0, min(H - kh, yi))

# def ensure_3level_xywh_inplace(df: pd.DataFrame) -> pd.DataFrame:
#     if df.columns.nlevels == 3:
#         return df
#     if df.columns.nlevels != 2:
#         raise ValueError("columns must have 2 or 3 levels.")
#     new_cols = []
#     for g, i in df.columns:
#         metric = "value" if g in META_TOPS else "xywh"
#         new_cols.append((g, i, metric))
#     df.columns = pd.MultiIndex.from_tuples(new_cols, names=["group","id","metric"])
#     return df

# def _collect_groups_ids(df: pd.DataFrame):
#     tops = [g for g in df.columns.get_level_values(0).unique() if g not in META_TOPS]
#     return {g: sorted({i for (g2,i,_) in df.columns if g2==g}) for g in tops}

# def _ensure_metric_columns(df: pd.DataFrame, gid_map, metrics=("density","centeredness","mixture")):
#     for g, ids in gid_map.items():
#         for i in ids:
#             for m in metrics:
#                 col = (g, i, m)
#                 if col not in df.columns:
#                     df[col] = np.nan

In [21]:
# def score_all_rows_fast(
#     merged: pd.DataFrame,
#     arr_getter,                    # callable(replay, image_id) -> np.ndarray (C,H,W)
#     kw: int = 20, kh: int = 12,
#     use_bbox: bool = False,
#     sigma=None,
#     mode: str = "entropy",
#     fill_when_const: float = 0.0, eps: float = 1e-6, pow_k: float = 1.0,
#     sort_for_locality: bool = True
# ):
#     df = merged
#     ensure_3level_xywh_inplace(df)

#     # 필요 컬럼 확보
#     gid_map = _collect_groups_ids(df)
#     _ensure_metric_columns(df, gid_map)

#     rcol = ("replay","", "value")
#     icol = ("image_id","", "value")
#     if rcol not in df.columns or icol not in df.columns:
#         raise ValueError("('replay','', 'value'), ('image_id','', 'value') 컬럼이 필요합니다.")

#     # 처리 순서: 디스크 지역성 향상
#     if sort_for_locality:
#         order = df.sort_values([rcol, icol]).index
#     else:
#         order = df.index

#     # --- 메인 루프 (적분영상 재사용) ---
#     for idx in order:
#         rep = df.at[idx, rcol]
#         img = df.at[idx, icol]
#         if pd.isna(rep) or pd.isna(img):
#             continue

#         X = arr_getter(rep, img)  # (C,H,W) or None
#         if X is None:
#             continue

#         x = X if X.ndim == 3 else X[None, ...]
#         if x.shape[0] <= 0:
#             # 채널이 0개면 스킵
#             continue

#         _, H, W = x.shape

#         # 이 행에서 필요한 커널 크기 모으기
#         ks_set = set()
#         if use_bbox:
#             for g, ids in gid_map.items():
#                 for i in ids:
#                     v = df.at[idx, (g, i, "xywh")] if (g,i,"xywh") in df.columns else np.nan
#                     xx,yy,ww,hh = _as_xywh(v)
#                     k_w = max(1, int(round(ww))) if np.isfinite(ww) else kw
#                     k_h = max(1, int(round(hh))) if np.isfinite(hh) else kh
#                     ks_set.add((min(k_w, W), min(k_h, H)))
#         else:
#             ks_set.add((kw, kh))

#         # 채널 합/분해 (float32 고정)
#         x_sum = x.sum(axis=0, dtype=np.float32)
#         xA = x[0:4, ...].sum(axis=0, dtype=np.float32)   # 필요시 슬라이스 조절
#         xB = x[4:8, ...].sum(axis=0, dtype=np.float32)

#         # 적분영상 한 번씩만
#         ii_sum = _integral_image(x_sum)
#         ii_A = _integral_image(xA)
#         ii_B = _integral_image(xB)

#         # 커널 크기별 맵 캐시
#         den_cache = {}
#         mix_cache = {}
#         cen_cache = {}

#         for k_w, k_h in ks_set:
#             # density: box 평균 → 안전 정규화
#             den = _box_sum(ii_sum, k_w, k_h) / float(k_w * k_h)
#             den_cache[(k_w,k_h)] = _safe_minmax_norm(den, fill=0.0)

#             # mixture: A,B 비율로 혼합도 → 안전 정규화
#             A = _box_sum(ii_A, k_w, k_h)
#             B = _box_sum(ii_B, k_w, k_h)
#             denAB = np.maximum(A + B, eps, dtype=np.float32)
#             p = A / denAB
#             if mode == 'entropy':
#                 p1 = np.clip(p, eps, 1.0 - eps); q1 = 1.0 - p1
#                 mix = -(p1*np.log(p1) + q1*np.log(q1)) / np.log(2.0)  # [0,1]
#             elif mode == 'gini':
#                 mix = 2.0 * p * (1.0 - p)                               # [0,0.5]
#             elif mode == 'linear':
#                 mix = 1.0 - np.abs(2.0*p - 1.0)                         # [0,1]
#             else:
#                 mix = 4.0 * p * (1.0 - p)                               # [0,1]
#             if pow_k != 1.0:
#                 mix = np.clip(mix, 0.0, 1.0) ** pow_k
#             mix_cache[(k_w,k_h)] = _safe_minmax_norm(mix, fill=fill_when_const)

#             # centeredness: 가우시안 컨볼브(valid)
#             cen_cache[(k_w,k_h)] = _centeredness_map(x_sum, k_w, k_h, sigma=sigma)

#         # 샘플링해서 df에 기록
#         for g, ids in gid_map.items():
#             for i in ids:
#                 v = df.at[idx, (g, i, "xywh")] if (g,i,"xywh") in df.columns else np.nan
#                 xx,yy,ww,hh = _as_xywh(v)
#                 k_w, k_h = ((kw,kh) if not use_bbox else
#                             (max(1, int(round(ww))) if np.isfinite(ww) else kw,
#                              max(1, int(round(hh))) if np.isfinite(hh) else kh))
#                 k_w = min(k_w, W); k_h = min(k_h, H)
#                 xi, yi = _clip_xy(xx, yy, W, H, k_w, k_h)

#                 df.at[idx, (g,i,"density")]     = float(den_cache[(k_w,k_h)][yi, xi])
#                 df.at[idx, (g,i,"centeredness")] = float(cen_cache[(k_w,k_h)][yi, xi])
#                 df.at[idx, (g,i,"mixture")]     = float(mix_cache[(k_w,k_h)][yi, xi])

In [22]:
# import os, math
# import numpy as np
# import pandas as pd
# from concurrent.futures import ThreadPoolExecutor, as_completed
# from functools import lru_cache
# from numpy.lib.stride_tricks import sliding_window_view

# META_TOPS = {"replay", "image_id"}

# # ---------- 유틸 ----------
# def _safe_minmax_norm(a: np.ndarray, fill=0.0) -> np.ndarray:
#     if a.size == 0: return np.empty_like(a, dtype=np.float32)
#     mn = np.nanmin(a); mx = np.nanmax(a); rng = mx - mn
#     if not np.isfinite(rng) or rng == 0:
#         return np.full_like(a, fill, dtype=np.float32)
#     out = (a - mn) / rng
#     return out.astype(np.float32, copy=False)

# def _integral_image(x2d: np.ndarray) -> np.ndarray:
#     ii = np.pad(x2d, ((1,0),(1,0)), mode='constant')
#     return ii.cumsum(axis=0).cumsum(axis=1)

# def _box_sum(ii: np.ndarray, kw: int, kh: int) -> np.ndarray:
#     return ii[kh:, kw:] - ii[:-kh, kw:] - ii[kh:, :-kw] + ii[:-kh, :-kw]

# @lru_cache(maxsize=128)
# def _gaussian_kernel2d_cached(kw: int, kh: int, sigma):
#     if sigma is None:
#         sx, sy = max(1e-6, kw/6.0), max(1e-6, kh/6.0)
#     elif np.isscalar(sigma):
#         sx = sy = float(sigma)
#     else:
#         sx, sy = float(sigma[0]), float(sigma[1])
#     x = np.arange(kw) - (kw - 1)/2.0
#     y = np.arange(kh) - (kh - 1)/2.0
#     X, Y = np.meshgrid(x, y)  # (kh,kw)
#     K = np.exp(-0.5*((X/(sx+1e-6))**2 + (Y/(sy+1e-6))**2)).astype(np.float32, copy=False)
#     s = K.sum();  K = K / (s if s>0 else 1.0)
#     return K.astype(np.float32, copy=False)

# def _centeredness_map(x_sum: np.ndarray, kw: int, kh: int, sigma=None) -> np.ndarray:
#     K = _gaussian_kernel2d_cached(kw, kh, None if sigma is None else tuple(np.atleast_1d(sigma)))
#     patches = sliding_window_view(x_sum, (kh, kw))  # (H-kh+1, W-kw+1, kh, kw)
#     cen = (patches * K).sum(axis=(-1, -2))
#     return _safe_minmax_norm(cen, fill=0.0)

# def _as_xywh(v):
#     if v is None: return (np.nan, np.nan, np.nan, np.nan)
#     if isinstance(v, dict):
#         for keys in (("x","y","w","h"), ("X","Y","W","H")):
#             if all(k in v for k in keys):
#                 return (float(v[keys[0]]), float(v[keys[1]]),
#                         float(v[keys[2]]), float(v[keys[3]]))
#         vals = list(v.values())
#         if len(vals) >= 4:
#             a = np.asarray(vals, dtype=float).ravel()
#             return tuple(a[:4])
#         return (np.nan, np.nan, np.nan, np.nan)
#     try:
#         a = np.asarray(v, dtype=float).ravel()
#         if a.size >= 4: return (float(a[0]), float(a[1]), float(a[2]), float(a[3]))
#     except Exception:
#         pass
#     return (np.nan, np.nan, np.nan, np.nan)

# def _clip_xy(xi, yi, W, H, kw, kh):
#     if not np.isfinite(xi): xi = 0
#     if not np.isfinite(yi): yi = 0
#     xi = int(round(xi)); yi = int(round(yi))
#     return max(0, min(W-kw, xi)), max(0, min(H-kh, yi))

# def ensure_3level_xywh_inplace(df: pd.DataFrame):
#     if df.columns.nlevels == 3: return df
#     if df.columns.nlevels != 2: raise ValueError("columns must have 2 or 3 levels.")
#     new_cols = []
#     for g,i in df.columns:
#         metric = "value" if g in META_TOPS else "xywh"
#         new_cols.append((g,i,metric))
#     df.columns = pd.MultiIndex.from_tuples(new_cols, names=["group","id","metric"])
#     return df

# def _collect_groups_ids(df: pd.DataFrame):
#     tops = [g for g in df.columns.get_level_values(0).unique() if g not in META_TOPS]
#     return {g: sorted({i for (g2,i,_) in df.columns if g2==g}) for g in tops}

# # ---------- I/O 프리패치 파이프라인 ----------
# def _prefetch_iter(index_order, df, rcol, icol, arr_getter, channel_slice, prefetch=128, io_workers=8):
#     """
#     (idx, x) 스트림 생성. x: (C,H,W) float32 또는 None.
#     arr_getter는 가능한 np.load(..., mmap_mode='r') 사용 추천.
#     """
#     with ThreadPoolExecutor(max_workers=io_workers) as ex:
#         it = iter(index_order)
#         inflight = {}
#         # 프라임
#         try:
#             for _ in range(prefetch):
#                 idx = next(it)
#                 rep = df.at[idx, rcol]; img = df.at[idx, icol]
#                 fut = ex.submit(arr_getter, rep, img)
#                 inflight[fut] = idx
#         except StopIteration:
#             pass

#         # 드레인
#         while inflight:
#             for fut in as_completed(list(inflight.keys())):
#                 idx = inflight.pop(fut)
#                 X = fut.result()
#                 if X is not None:
#                     x = X if X.ndim==3 else X[None,...]
#                     if channel_slice is not None:
#                         x = x[channel_slice,:,:]
#                     yield idx, x.astype(np.float32, copy=False)
#                 else:
#                     yield idx, None
#                 # 보충
#                 try:
#                     nxt = next(it)
#                     rep = df.at[nxt, rcol]; img = df.at[nxt, icol]
#                     inflight[ex.submit(arr_getter, rep, img)] = nxt
#                 except StopIteration:
#                     pass

# # ---------- 메인: 빠른 일괄 계산 + 컬럼 단위 할당 ----------
# def score_all_rows_pipeline(
#     merged: pd.DataFrame,
#     arr_getter,                    # (replay,image_id)-> np.ndarray(C,H,W) or None
#     channel_slice=None,            # 예: slice(0, Channel.Player_2_Building.value)
#     kw: int = 20, kh: int = 12,
#     use_bbox: bool = False,
#     sigma=None,
#     mode: str = "entropy",
#     fill_when_const: float = 0.0, eps: float = 1e-6, pow_k: float = 1.0,
#     io_workers: int = 8, prefetch: int = 128, sort_for_locality: bool = True
# ):
#     df = merged
#     ensure_3level_xywh_inplace(df)
#     gid_map = _collect_groups_ids(df)

#     # 미리 필요한 결과 컬럼 생성 + 결과 버퍼 준비 (행 길이만큼)
#     metrics = ("density","centeredness","mixture")
#     rcol = ("replay","", "value")
#     icol = ("image_id","", "value")
#     order = df.sort_values([rcol, icol]).index if sort_for_locality else df.index
#     pos_of = {idx:pos for pos,idx in enumerate(order)}
#     n = len(order)

#     out = {}  # (group,id,metric) -> np.array(n,)
#     for g, ids in gid_map.items():
#         for i in ids:
#             for m in metrics:
#                 col = (g,i,m)
#                 if col not in df.columns:
#                     df[col] = np.nan
#                 out[col] = np.full(n, np.nan, dtype=np.float32)

#     # I/O–컴퓨트 파이프라인
#     for idx, x in _prefetch_iter(order, df, rcol, icol, arr_getter, channel_slice,
#                                  prefetch=prefetch, io_workers=io_workers):
#         pos = pos_of[idx]
#         if x is None or x.shape[0] == 0:
#             continue

#         _, H, W = x.shape
#         # 이 행에서 필요한 커널 크기 수집
#         ks_set = set()
#         if use_bbox:
#             for g, ids in gid_map.items():
#                 for i in ids:
#                     v = df.at[idx, (g,i,"xywh")] if (g,i,"xywh") in df.columns else np.nan
#                     xx,yy,ww,hh = _as_xywh(v)
#                     k_w = max(1, int(round(ww))) if np.isfinite(ww) else kw
#                     k_h = max(1, int(round(hh))) if np.isfinite(hh) else kh
#                     ks_set.add((min(k_w,W), min(k_h,H)))
#         else:
#             ks_set.add((kw,kh))

#         # 채널 합/분해 + 적분영상 (행당 1회씩만)
#         x_sum = x.sum(axis=0, dtype=np.float32)
#         ii_sum = _integral_image(x_sum)

#         # mixture용 (필요하면 슬라이스 조정)
#         xA = x[0:4,...].sum(axis=0, dtype=np.float32)
#         xB = x[4:8,...].sum(axis=0, dtype=np.float32)
#         ii_A = _integral_image(xA)
#         ii_B = _integral_image(xB)

#         den_cache = {}
#         mix_cache = {}
#         cen_cache = {}
#         for k_w, k_h in ks_set:
#             den = _box_sum(ii_sum, k_w, k_h) / float(k_w * k_h)
#             den_cache[(k_w,k_h)] = _safe_minmax_norm(den, fill=0.0)

#             A = _box_sum(ii_A, k_w, k_h); B = _box_sum(ii_B, k_w, k_h)
#             denAB = np.maximum(A + B, eps, dtype=np.float32)
#             p = A / denAB
#             if mode == 'entropy':
#                 p1 = np.clip(p, eps, 1.0 - eps); q1 = 1.0 - p1
#                 mix = -(p1*np.log(p1) + q1*np.log(q1)) / np.log(2.0)
#             elif mode == 'gini':
#                 mix = 2.0*p*(1.0-p)
#             elif mode == 'linear':
#                 mix = 1.0 - np.abs(2.0*p - 1.0)
#             else:
#                 mix = 4.0*p*(1.0-p)
#             if pow_k != 1.0:
#                 mix = np.clip(mix, 0.0, 1.0) ** pow_k
#             mix_cache[(k_w,k_h)] = _safe_minmax_norm(mix, fill=fill_when_const)

#             cen_cache[(k_w,k_h)] = _centeredness_map(x_sum, k_w, k_h, sigma=sigma)

#         # 한 번에 샘플링해서 버퍼에 기록 (셀별 df.at 금지)
#         for g, ids in gid_map.items():
#             for i in ids:
#                 v = df.at[idx, (g,i,"xywh")] if (g,i,"xywh") in df.columns else np.nan
#                 xx,yy,ww,hh = _as_xywh(v)
#                 k_w, k_h = ((kw,kh) if not use_bbox else
#                             (max(1,int(round(ww))) if np.isfinite(ww) else kw,
#                              max(1,int(round(hh))) if np.isfinite(hh) else kh))
#                 k_w = min(k_w, W); k_h = min(k_h, H)
#                 xi, yi = _clip_xy(xx, yy, W, H, k_w, k_h)
#                 out[(g,i,"density")][pos]      = den_cache[(k_w,k_h)][yi, xi]
#                 out[(g,i,"centeredness")][pos] = cen_cache[(k_w,k_h)][yi, xi]
#                 out[(g,i,"mixture")][pos]      = mix_cache[(k_w,k_h)][yi, xi]

#     # 컬럼 단위 일괄 할당 (매우 빠름)
#     for col, buf in out.items():
#         df.loc[order, col] = buf


In [127]:
def arr_getter(replay, image_id):
    if not str(replay).endswith('.rep'):
        replay = str(replay) + '.rep'
    path = f"/workspace/data/input/dst/{str(replay)}/{int(image_id)}.npy"
    try:
        arr = np.load(path, mmap_mode='r')  # (C,H,W)
        return arr.astype(np.float32, copy=False)
    except Exception:
        return None

In [145]:
import os, math
import numpy as np
import pandas as pd
import multiprocessing as mp
from numpy.lib.stride_tricks import sliding_window_view

# ---------- 튜닝 파라미터 ----------
WORKERS = 16 # max(1, int(mp.cpu_count() * 0.75))  # I/O면 4~8도 충분
BATCH_SIZE = 512                                # 한 번에 처리할 행 수 (IPC 감소)
KW, KH = 20, 12
USE_BBOX = False
SIGMA = None
MODE = "entropy"
EPS = 1e-6
POW_K = 1.0

# ---------- 공용 유틸 ----------
META_TOPS = {"replay", "image_id"}

def ensure_3level_xywh_inplace(df: pd.DataFrame):
    if df.columns.nlevels == 3: return df
    new_cols = []
    for g, i in df.columns:
        metric = "value" if g in META_TOPS else "xywh"
        new_cols.append((g, i, metric))
    df.columns = pd.MultiIndex.from_tuples(new_cols, names=["group","id","metric"])
    return df

def collect_groups_ids(df: pd.DataFrame):
    tops = [g for g in df.columns.get_level_values(0).unique() if g not in META_TOPS]
    return {g: sorted({i for (g2,i,_) in df.columns if g2==g}) for g in tops}

def as_xywh(v):
    if v is None: return (np.nan, np.nan, np.nan, np.nan)
    if isinstance(v, dict):
        for keys in (("x","y","w","h"), ("X","Y","W","H")):
            if all(k in v for k in keys):
                return (float(v[keys[0]]), float(v[keys[1]]),
                        float(v[keys[2]]), float(v[keys[3]]))
        vals = list(v.values())
        if len(vals) >= 4:
            a = np.asarray(vals, dtype=float).ravel()
            return tuple(a[:4])
        return (np.nan, np.nan, np.nan, np.nan)
    try:
        a = np.asarray(v, dtype=float).ravel()
        if a.size >= 4:
            return (float(a[0]), float(a[1]), float(a[2]), float(a[3]))
    except Exception:
        pass
    return (np.nan, np.nan, np.nan, np.nan)

def safe_minmax(a, fill=0.0):
    if a.size == 0: return np.empty_like(a, dtype=np.float32)
    mn = np.nanmin(a); mx = np.nanmax(a); rng = mx - mn
    if not np.isfinite(rng) or rng == 0:
        return np.full_like(a, fill, dtype=np.float32)
    out = (a - mn) / rng
    return out.astype(np.float32, copy=False)

def integral(x2d):
    ii = np.pad(x2d, ((1,0),(1,0)), mode='constant')
    return ii.cumsum(axis=0).cumsum(axis=1)

def box_sum(ii, kw, kh):
    return ii[kh:, kw:] - ii[:-kh, kw:] - ii[kh:, :-kw] + ii[:-kh, :-kw]

def centeredness_map(x_sum, kw, kh, sigma=None):
    # 간단한 gaussian kernel (sum=1) + sliding window
    if sigma is None:
        sx, sy = max(1e-6, kw/6.0), max(1e-6, kh/6.0)
    elif np.isscalar(sigma):
        sx = sy = float(sigma)
    else:
        sx, sy = float(sigma[0]), float(sigma[1])
    x = np.arange(kw) - (kw - 1)/2.0
    y = np.arange(kh) - (kh - 1)/2.0
    X, Y = np.meshgrid(x, y)
    K = np.exp(-0.5*((X/(sx+1e-6))**2 + (Y/(sy+1e-6))**2)).astype(np.float32, copy=False)
    s = K.sum();  K = K / (s if s > 0 else 1.0)
    patches = sliding_window_view(x_sum, (kh, kw))
    cen = (patches * K).sum(axis=(-1, -2))
    return safe_minmax(cen, fill=0.0)

def clip_xy(xi, yi, W, H, kw, kh):
    if not np.isfinite(xi): xi = 0
    if not np.isfinite(yi): yi = 0
    xi = int(round(xi)); yi = int(round(yi))
    return max(0, min(W-kw, xi)), max(0, min(H-kh, yi))

# ---------- arr_getter (워커에서 쓸 전역) ----------
ARR_GETTER = None
CHANNEL_SLICE = None
GID_MAP = None

def init_worker(arr_getter, channel_slice, gid_map):
    # 과도한 내부 스레드 제한 (중첩 방지)
    os.environ.setdefault("OMP_NUM_THREADS", "1")
    os.environ.setdefault("MKL_NUM_THREADS", "1")
    os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
    global ARR_GETTER, CHANNEL_SLICE, GID_MAP
    ARR_GETTER = arr_getter
    CHANNEL_SLICE = channel_slice
    GID_MAP = gid_map

In [146]:
# ---------- 워커: 배치 처리 ----------
def worker_batch(tasks):
    """
    tasks: list of (pos, replay, image_id, xywh_dict)
           xywh_dict: { (group,id): (x,y,w,h), ... }
    return: list of (pos, { (g,i,'density'):v, (g,i,'centeredness'):v, (g,i,'mixture'):v, ... })
    """
    out = []
    skip = 0
    for pos, rep, img, xywh_d in tasks:
        X = ARR_GETTER(rep, img)
        if X is None:
            skip += 1
            out.append((pos, {}))
            continue
        x = X if X.ndim == 3 else X[None, ...]
        if CHANNEL_SLICE is not None:
            x = x[CHANNEL_SLICE, :, :]
        if x.shape[0] == 0:
            skip += 1
            out.append((pos, {}))
            continue

        _, H, W = x.shape

        # 커널 크기 수집
        ks = set()
        if USE_BBOX:
            for (g,i), (xx,yy,ww,hh) in xywh_d.items():
                k_w = max(1, int(round(ww))) if np.isfinite(ww) else KW
                k_h = max(1, int(round(hh))) if np.isfinite(hh) else KH
                ks.add((min(k_w, W), min(k_h, H)))
        else:
            ks.add((KW, KH))

        # 채널 합 & 적분영상 (행당 1회)
        x_sum = x.sum(axis=0, dtype=np.float32)
        ii_sum = integral(x_sum)

        xA = x[0:4, ...].sum(axis=0, dtype=np.float32)
        xB = x[4:8, ...].sum(axis=0, dtype=np.float32)
        ii_A = integral(xA)
        ii_B = integral(xB)

        den_cache, mix_cache, cen_cache = {}, {}, {}
        for kw_, kh_ in ks:
            den = box_sum(ii_sum, kw_, kh_) / float(kw_ * kh_)
            den_cache[(kw_,kh_)] = safe_minmax(den, fill=0.0)

            A = box_sum(ii_A, kw_, kh_)
            B = box_sum(ii_B, kw_, kh_)
            denAB = np.maximum(A + B, EPS, dtype=np.float32)
            p = A / denAB
            if MODE == 'entropy':
                p1 = np.clip(p, EPS, 1.0 - EPS); q1 = 1.0 - p1
                mix = -(p1*np.log(p1) + q1*np.log(q1)) / np.log(2.0)
            elif MODE == 'gini':
                mix = 2.0 * p * (1.0 - p)
            elif MODE == 'linear':
                mix = 1.0 - np.abs(2.0*p - 1.0)
            else:
                mix = 4.0 * p * (1.0 - p)
            if POW_K != 1.0:
                mix = np.clip(mix, 0.0, 1.0) ** POW_K
            mix_cache[(kw_,kh_)] = safe_minmax(mix, fill=0.0)

            cen_cache[(kw_,kh_)] = centeredness_map(x_sum, kw_, kh_, sigma=SIGMA)

        row_res = {}
        for (g, i), (xx,yy,ww,hh) in xywh_d.items():
            kw_, kh_ = ((KW,KH) if not USE_BBOX else
                        (max(1,int(round(ww))) if np.isfinite(ww) else KW,
                         max(1,int(round(hh))) if np.isfinite(hh) else KH))
            kw_, kh_ = min(kw_, W), min(kh_, H)
            xi, yi = clip_xy(xx, yy, W, H, kw_, kh_)
            row_res[(g,i,'density')]      = float(den_cache[(kw_,kh_)][yi, xi])
            row_res[(g,i,'centeredness')] = float(cen_cache[(kw_,kh_)][yi, xi])
            row_res[(g,i,'mixture')]      = float(mix_cache[(kw_,kh_)][yi, xi])

        out.append((pos, row_res))

    ok = len(tasks) - skip
    
    return out, ok, skip

In [147]:
def worker_batch_with_id(args):
    batch_id, tasks = args
    out, ok, skip = worker_batch(tasks)
    return batch_id, out, ok, skip

In [148]:
from tqdm.auto import tqdm

In [149]:
def score_all_rows_mp_with_progress(merged: pd.DataFrame,
                                    arr_getter,
                                    channel_slice=None,
                                    workers=None,
                                    batch_size=64,
                                    desc="Scoring"):
    df = merged
    ensure_3level_xywh_inplace(df)

    gid_map = collect_groups_ids(df)
    metrics = ("density","centeredness","mixture")
    rcol = ("replay","", "value")
    icol = ("image_id","", "value")
    order = df.sort_values([rcol, icol]).index
    pos_of = {idx: pos for pos, idx in enumerate(order)}
    n = len(order)

    # 결과 버퍼
    out_buf = {}
    for g, ids in gid_map.items():
        for i in ids:
            for m in metrics:
                col = (g,i,m)
                if col not in df.columns:
                    df[col] = np.nan
                out_buf[col] = np.full(n, np.nan, dtype=np.float32)

    # 태스크 만들기
    tasks = []
    for idx in order:
        rep = df.at[idx, rcol]; img = df.at[idx, icol]
        xywh_d = {}
        for g, ids in gid_map.items():
            for i in ids:
                v = df.at[idx, (g,i,"xywh")] if (g,i,"xywh") in df.columns else np.nan
                xywh_d[(g,i)] = as_xywh(v)
        tasks.append((pos_of[idx], rep, img, xywh_d))

    # 배치 분할 + 사이즈 기록
    B = batch_size
    batches = [tasks[i:i+B] for i in range(0, len(tasks), B)]
    batch_sizes = [len(b) for b in batches]
    total_rows = sum(batch_sizes)

    # 풀 시작 (spawn/fork 환경 둘 다 지원)
    ctx = mp.get_context("spawn") if mp.get_start_method(allow_none=True) != "fork" else mp.get_context("fork")
    if workers is None:
        workers = max(1, int(mp.cpu_count() * 0.75))

    print(f"use {workers} workers")
    
    with ctx.Pool(processes=workers,
                  initializer=init_worker,
                  initargs=(arr_getter, channel_slice, gid_map)) as pool, \
         tqdm(total=total_rows, desc=desc, unit="row") as pbar:

        # 배치 ID 부여해서 전달
        total_ok = total_skip = 0
        for batch_id, batch_res, ok, skip in pool.imap_unordered(worker_batch_with_id, enumerate(batches), chunksize=1):
            total_ok  += ok
            total_skip += skip
            
            pbar.update(batch_sizes[batch_id])
            # 결과 반영
            for pos, row_res in batch_res:
                if not row_res:
                    continue
                for col, val in row_res.items():
                    out_buf[col][pos] = val
            pbar.set_postfix(ok=total_ok, skip=total_skip)

    # 한 번에 df에 기록
    for col, buf in out_buf.items():
        df.loc[order, col] = buf

In [150]:
for d in [ground_truth_set1, inference_vanilla_set1, inference_kbrs_set1]:
    assert not d.duplicated(['replay','image_id']).any()

gt = (ground_truth_set1
      .set_index(['replay','image_id'])
      .rename(columns=lambda c: c.replace('gt_', '')))

van = (inference_vanilla_set1
       .set_index(['replay','image_id'])
       .rename(columns=lambda c: c.replace('pred_', '')))

kbrs = (inference_kbrs_set1
        .set_index(['replay','image_id'])
        .rename(columns=lambda c: c.replace('pred_', '')))

# 축으로 합치며 상위 레벨 키 부여
merged = pd.concat({'gt': gt, 'vanilla': van, 'kbrs': kbrs}, axis=1)
merged = merged.sort_index(axis=1)  # 보기 좋게 정렬

# 정렬 보장
merged = merged.sort_index(level=["replay", "image_id"])

# kbrs / vanilla 블록만 채우기 (gt는 그대로)
idx = pd.IndexSlice
cols_to_bfill = merged.loc[:, idx[["kbrs", "vanilla"], :]].columns

merged.loc[:, cols_to_bfill] = (
    merged.loc[:, cols_to_bfill]
      .groupby(level="replay", as_index=False)   # 같은 replay 안에서만
      .bfill()                   # 다음 프레임의 값으로 메움
)

merged = merged.dropna()
merged = merged.reset_index()

In [151]:
# # 모든 행 처리 (채널 슬라이스는 예: Player_2_Building 미만 채널만 사용)
# score_all_rows(
#     merged,
#     arr_getter=arr_getter,
#     kw=20, kh=12,
#     use_bbox=False,           # True면 xywh별 (w,h) 사용
#     sigma=None,
#     mode="entropy",
#     pow_k=1.0,
#     parallel_workers=32        # IO 비중이 크면 ThreadPool이 유리
# )

In [152]:
score_all_rows_mp_with_progress(
    merged,
    arr_getter=arr_getter,
    workers=WORKERS,
)

use 16 workers


Scoring:   0%|          | 0/148659 [00:00<?, ?row/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [ ]:
merged.to_csv('result.csv')

In [154]:
merged

group     replay image_id                gt                    \
id                                        0                 1   
metric     value    value              xywh              xywh   
0       1725.rep        0  [108, 2, 20, 12]  [108, 2, 20, 12]   
1       1725.rep        1  [108, 2, 20, 12]  [108, 2, 20, 12]   
2       1725.rep        2  [108, 2, 20, 12]  [108, 2, 20, 12]   
3       1725.rep        3  [108, 2, 20, 12]  [108, 2, 20, 12]   
4       1725.rep        4  [108, 2, 20, 12]  [108, 2, 20, 12]   
...          ...      ...               ...               ...   
148654  4664.rep    20629  [89, 83, 20, 12]  [95, 84, 20, 12]   
148655  4664.rep    20630  [89, 83, 20, 12]  [95, 84, 20, 12]   
148656  4664.rep    20631  [89, 83, 20, 12]  [95, 84, 20, 12]   
148657  4664.rep    20632  [89, 83, 20, 12]  [95, 84, 20, 12]   
148658  4664.rep    20633  [89, 83, 20, 12]  [98, 84, 20, 12]   

group                                                          \
id                     2                 3                  4   
metric              xywh              xywh               xywh   
0       [108, 2, 20, 12]  [108, 2, 20, 12]   [108, 2, 20, 12]   
1       [108, 2, 20, 12]  [108, 2, 20, 12]   [108, 2, 20, 12]   
2       [108, 2, 20, 12]  [108, 2, 20, 12]   [108, 2, 20, 12]   
3       [108, 2, 20, 12]  [108, 2, 20, 12]   [108, 2, 20, 12]   
4       [108, 2, 20, 12]  [108, 2, 20, 12]   [108, 2, 20, 12]   
...                  ...               ...                ...   
148654  [90, 83, 20, 12]  [92, 86, 20, 12]  [102, 85, 20, 12]   
148655  [90, 83, 20, 12]  [92, 86, 20, 12]  [102, 85, 20, 12]   
148656  [90, 83, 20, 12]  [92, 86, 20, 12]  [102, 85, 20, 12]   
148657  [90, 83, 20, 12]  [92, 86, 20, 12]  [102, 85, 20, 12]   
148658  [90, 83, 20, 12]  [92, 86, 20, 12]  [102, 85, 20, 12]   

group               kbrs                                          ... vanilla  \
id                     0                   1                   2  ...       1   
metric              xywh                xywh                xywh  ... mixture   
0         [0, 2, 19, 12]  [107, 112, 20, 12]      [2, 5, 20, 12]  ...     1.0   
1         [0, 2, 19, 12]  [107, 112, 20, 12]      [2, 5, 20, 12]  ...     1.0   
2         [0, 2, 19, 12]  [107, 112, 20, 12]      [2, 5, 20, 12]  ...     1.0   
3         [0, 2, 19, 12]  [107, 112, 20, 12]      [2, 5, 20, 12]  ...     1.0   
4         [0, 2, 19, 12]  [107, 112, 20, 12]      [2, 5, 20, 12]  ...     1.0   
...                  ...                 ...                 ...  ...     ...   
148654  [95, 84, 20, 12]    [90, 82, 20, 12]  [107, 113, 20, 12]  ...     0.0   
148655  [95, 84, 20, 12]    [90, 82, 20, 12]  [107, 113, 20, 12]  ...     0.0   
148656  [95, 84, 20, 12]    [88, 83, 20, 12]  [107, 113, 20, 12]  ...     0.0   
148657  [95, 84, 20, 12]    [88, 83, 20, 12]  [107, 113, 20, 12]  ...     0.0   
148658  [95, 84, 20, 12]    [88, 83, 20, 12]  [107, 113, 20, 12]  ...     0.0   

group                                                                       \
id             2                              3                              
metric   density centeredness mixture   density centeredness       mixture   
0       0.994565     0.996460     1.0  0.664402     0.640182  0.000000e+00   
1       0.994565     0.996460     1.0  0.664402     0.640182  0.000000e+00   
2       0.994565     0.996460     1.0  0.664402     0.640182  0.000000e+00   
3       0.994565     0.996460     1.0  0.664402     0.640182  0.000000e+00   
4       0.994565     0.996460     1.0  0.664402     0.640182  0.000000e+00   
...          ...          ...     ...       ...          ...           ...   
148654  0.997468     0.971048     0.0  0.408861     0.396934  4.689843e-01   
148655  0.997468     0.971048     0.0  0.967089     0.938184  0.000000e+00   
148656  1.000000     0.963390     0.0  0.744304     0.871512  2.453907e-07   
148657  0.926489     0.917168     0.0  0.745247     0.870865  2.453907e-07   
148658  0.926489     0.918176

In [282]:
# merged.columns

In [155]:
merged[[(  'replay',  '',        'value'),
        ('image_id',  '',        'value'),
        (      'gt', '0',      'density'),
        (      'gt', '0', 'centeredness'),
        (      'gt', '0',      'mixture'),
        (      'gt', '1',      'density'),
        (      'gt', '1', 'centeredness'),
        (      'gt', '1',      'mixture'),
        (      'gt', '2',      'density'),
        (      'gt', '2', 'centeredness'),
        (      'gt', '2',      'mixture'),
        (      'gt', '3',      'density'),
        (      'gt', '3', 'centeredness'),
        (      'gt', '3',      'mixture'),
        (      'gt', '4',      'density'),
        (      'gt', '4', 'centeredness'),
        (      'gt', '4',      'mixture'),
        (    'kbrs', '0',      'density'),
        (    'kbrs', '0', 'centeredness'),
        (    'kbrs', '0',      'mixture'),
        (    'kbrs', '1',      'density'),
        (    'kbrs', '1', 'centeredness'),
        (    'kbrs', '1',      'mixture'),
        (    'kbrs', '2',      'density'),
        (    'kbrs', '2', 'centeredness'),
        (    'kbrs', '2',      'mixture'),
        (    'kbrs', '3',      'density'),
        (    'kbrs', '3', 'centeredness'),
        (    'kbrs', '3',      'mixture'),
        (    'kbrs', '4',      'density'),
        (    'kbrs', '4', 'centeredness'),
        (    'kbrs', '4',      'mixture'),
        ( 'vanilla', '0',      'density'),
        ( 'vanilla', '0', 'centeredness'),
        ( 'vanilla', '0',      'mixture'),
        ( 'vanilla', '1',      'density'),
        ( 'vanilla', '1', 'centeredness'),
        ( 'vanilla', '1',      'mixture'),
        ( 'vanilla', '2',      'density'),
        ( 'vanilla', '2', 'centeredness'),
        ( 'vanilla', '2',      'mixture'),
        ( 'vanilla', '3',      'density'),
        ( 'vanilla', '3', 'centeredness'),
        ( 'vanilla', '3',      'mixture'),
        ( 'vanilla', '4',      'density'),
        ( 'vanilla', '4', 'centeredness'),
        ( 'vanilla', '4',      'mixture'),
       ]]

group     replay image_id        gt                                   \
id                                0                                1   
metric     value    value   density centeredness   mixture   density   
0       1725.rep        0  0.665761     0.640732  0.000000  0.665761   
1       1725.rep        1  0.665761     0.640732  0.000000  0.665761   
2       1725.rep        2  0.665761     0.640732  0.000000  0.665761   
3       1725.rep        3  0.665761     0.640732  0.000000  0.665761   
4       1725.rep        4  0.665761     0.640732  0.000000  0.665761   
...          ...      ...       ...          ...       ...       ...   
148654  4664.rep    20629  0.394937     0.396129  0.391231  0.393671   
148655  4664.rep    20630  0.394937     0.395539  0.391231  0.393671   
148656  4664.rep    20631  0.394937     0.395554  0.391231  0.394937   
148657  4664.rep    20632  0.395437     0.394682  0.391231  0.395437   
148658  4664.rep    20633  0.395437     0.394738  0.391231  0.416984   

group                                                 ... vanilla            \
id                                    2               ...       1         2   
metric centeredness   mixture   density centeredness  ... mixture   density   
0          0.640732  0.000000  0.665761     0.640732  ...     1.0  0.994565   
1          0.640732  0.000000  0.665761     0.640732  ...     1.0  0.994565   
2          0.640732  0.000000  0.665761     0.640732  ...     1.0  0.994565   
3          0.640732  0.000000  0.665761     0.640732  ...     1.0  0.994565   
4          0.640732  0.000000  0.665761     0.640732  ...     1.0  0.994565   
...             ...       ...       ...          ...  ...     ...       ...   
148654     0.401324  0.934067  0.400000     0.410422  ...     0.0  0.997468   
148655     0.402079  0.934067  0.400000     0.410334  ...     0.0  0.997468   
148656     0.404012  0.926211  0.400000     0.410410  ...     0.0  1.000000   
148657     0.402193  0.926211  0.400507     0.409201  ...     0.0  0.926489   
148658     0.364138  0.988283  0.400507     0.409430  ...     0.0  0.926489   

group                                                                       \
id                                  3                                    4   
metric centeredness mixture   density centeredness       mixture   density   
0          0.996460     1.0  0.664402     0.640182  0.000000e+00  0.963315   
1          0.996460     1.0  0.664402     0.640182  0.000000e+00  0.963315   
2          0.996460     1.0  0.664402     0.640182  0.000000e+00  0.963315   
3          0.996460     1.0  0.664402     0.640182  0.000000e+00  0.963315   
4          0.996460     1.0  0.664402     0.640182  0.000000e+00  0.963315   
...             ...     ...       ...          ...           ...       ...   
148654     0.971048     0.0  0.408861     0.396934  4.689843e-01  0.967089   
148655     0.971048     0.0  0.967089     0.938184  0.000000e+00  0.744304   
148656     0.963390     0.0  0.744304     0.871512  2.453907e-07  0.329114   
148657     0.917168     0.0  0.745247     0.870865  2.453907e-07  0.329531   
148658     0.918176     0.0  0.745247     0.871822  2.453907e-07  0.329531   

group                              
id                                 
metric centeredness       mixture  
0          0.962158  0.000000e+00  
1          0.962158  0.000000e+00  
2          0.962158  0.000000e+00  
3          0.962158  0.000000e+00  
4          0.962158  0.000000e+00  
...             ...           ...  
148654     0.938184  0.000000e+00  
148655     0.871512  2.453907e-07  
148656     0.324509  2.453907e-07  
148657     0.324269  2.453907e-07  
148658     0.324625  2.453907e-07  

[148659 rows x 47 columns]

In [157]:
cols_gt_metric = [
    (      "gt", "0",      "density"),
    (      "gt", "0", "centeredness"),
    (      "gt", "0",      "mixture"),
    (      "gt", "1",      "density"),
    (      "gt", "1", "centeredness"),
    (      "gt", "1",      "mixture"),
    (      "gt", "2",      "density"),
    (      "gt", "2", "centeredness"),
    (      "gt", "2",      "mixture"),
    (      "gt", "3",      "density"),
    (      "gt", "3", "centeredness"),
    (      "gt", "3",      "mixture"),
    (      "gt", "4",      "density"),
    (      "gt", "4", "centeredness"),
    (      "gt", "4",      "mixture"),
]

cols_vanilla_metric = [
    ( "vanilla", "0",      "density"),
    ( "vanilla", "0", "centeredness"),
    ( "vanilla", "0",      "mixture"),
    ( "vanilla", "1",      "density"),
    ( "vanilla", "1", "centeredness"),
    ( "vanilla", "1",      "mixture"),
    ( "vanilla", "2",      "density"),
    ( "vanilla", "2", "centeredness"),
    ( "vanilla", "2",      "mixture"),
    ( "vanilla", "3",      "density"),
    ( "vanilla", "3", "centeredness"),
    ( "vanilla", "3",      "mixture"),
    ( "vanilla", "4",      "density"),
    ( "vanilla", "4", "centeredness"),
    ( "vanilla", "4",      "mixture"),
]

cols_kbrs_metric = [
    (    "kbrs", "0",      "density"),
    (    "kbrs", "0", "centeredness"),
    (    "kbrs", "0",      "mixture"),
    (    "kbrs", "1",      "density"),
    (    "kbrs", "1", "centeredness"),
    (    "kbrs", "1",      "mixture"),
    (    "kbrs", "2",      "density"),
    (    "kbrs", "2", "centeredness"),
    (    "kbrs", "2",      "mixture"),
    (    "kbrs", "3",      "density"),
    (    "kbrs", "3", "centeredness"),
    (    "kbrs", "3",      "mixture"),
    (    "kbrs", "4",      "density"),
    (    "kbrs", "4", "centeredness"),
    (    "kbrs", "4",      "mixture"),
]

In [161]:
gt_merged = merged[cols_gt_metric]
vanilla_merged = merged[cols_vanilla_metric]
kbrs_merged = merged[cols_kbrs_metric]

In [162]:
metric_desc_overall_gt = (
    gt_merged.T.groupby(level="metric").apply(lambda x: pd.Series(x.to_numpy().ravel()).describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
).T
metric_desc_overall_vanilla = (
    vanilla_merged.T.groupby(level="metric").apply(lambda x: pd.Series(x.to_numpy().ravel()).describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
).T
metric_desc_overall_kbrs = (
    kbrs_merged.T.groupby(level="metric").apply(lambda x: pd.Series(x.to_numpy().ravel()).describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
).T

In [163]:
metric_order = ["centeredness", "density", "mixture"]

gt = metric_desc_overall_gt.reindex(columns=metric_order)
van = metric_desc_overall_vanilla.reindex(columns=metric_order)
kbrs = metric_desc_overall_kbrs.reindex(columns=metric_order)

merged_overall = pd.concat({"gt": gt, "vanilla": van, "kbrs": kbrs}, axis=1)
merged_overall.columns.names = ["group", "metric"]
merged_overall.index.name = "stat"   # count, mean, std, ... 행 이름

In [164]:
merged_overall

group              gt                                     vanilla  \
metric   centeredness        density       mixture   centeredness   
stat                                                                
count   743295.000000  743295.000000  7.432950e+05  743295.000000   
mean         0.591878       0.595750  3.885415e-01       0.642109   
std          0.287001       0.278799  4.351161e-01       0.293292   
min          0.000000       0.000000  0.000000e+00       0.006069   
50%          0.411437       0.445026  2.453907e-07       0.459924   
90%          0.990914       0.993261  1.000000e+00       0.989586   
95%          0.997296       0.997319  1.000000e+00       0.996142   
99%          1.000000       1.000000  1.000000e+00       1.000000   
max          1.000000       1.000000  1.000000e+00       1.000000   

group                                         kbrs                 \
metric        density       mixture   centeredness        density   
stat                                                                
count   743295.000000  7.432950e+05  743295.000000  743295.000000   
mean         0.647339  3.040103e-01       0.614447       0.618186   
std          0.283007  4.152294e-01       0.290406       0.281098   
min          0.031746  0.000000e+00       0.073043       0.143826   
50%          0.495289  2.453907e-07       0.415271       0.451656   
90%          0.993378  9.975024e-01       0.988969       0.992021   
95%          0.997312  1.000000e+00       0.996179       0.995973   
99%          1.000000  1.000000e+00       1.000000       1.000000   
max          1.000000  1.000000e+00       1.000000       1.000000   

group                 
metric       mixture  
stat                  
count   7.432950e+05  
mean    3.519961e-01  
std     4.231127e-01  
min     0.000000e+00  
50%     2.453907e-07  
90%     9.986359e-01  
95%     1.000000e+00  
99%     1.000000e+00  
max     1.000000e+00

In [166]:
targets_gt = [
    ("mean", ("gt","centeredness")),
    ("mean", ("gt","density")),
    ("mean", ("gt","mixture")),
]
targets_van = [
    ("mean", ("vanilla","centeredness")),
    ("mean", ("vanilla","density")),
    ("mean", ("vanilla","mixture")),
]
targets_kbrs = [
    ("mean", ("kbrs","centeredness")),
    ("mean", ("kbrs","density")),
    ("mean", ("kbrs","mixture")),
]

sty = merged_overall.style
for r, c in targets_gt:
    sty = sty.set_properties(subset=([r], [c]),
                             **{"background-color":"#d7f9d7","font-weight":"600"})
for r, c in targets_van:
    sty = sty.set_properties(subset=([r], [c]),
                             **{"background-color":"#c7e9ff","font-weight":"600"})
for r, c in targets_kbrs:
    sty = sty.set_properties(subset=([r], [c]),
                             **{"background-color":"#ffe3d8","font-weight":"600"})
sty

In [167]:
tmp = merged[[    (      "gt", "0",      "xywh"),
    (      "gt", "1",      "xywh"),
    (      "gt", "2",      "xywh"),
    (      "gt", "3",      "xywh"),
    (      "gt", "4",      "xywh"),
    (      "kbrs", "0",      "xywh"),
    (      "kbrs", "1",      "xywh"),
    (      "kbrs", "2",      "xywh"),
    (      "kbrs", "3",      "xywh"),
    (      "kbrs", "4",      "xywh"),
    (      "vanilla", "0",      "xywh"),
    (      "vanilla", "1",      "xywh"),
    (      "vanilla", "2",      "xywh"),
    (      "vanilla", "3",      "xywh"),
    (      "vanilla", "4",      "xywh"),]]

tmp = tmp.dropna()
tmp.columns

MultiIndex([(     'gt', '0', 'xywh'),
            (     'gt', '1', 'xywh'),
            (     'gt', '2', 'xywh'),
            (     'gt', '3', 'xywh'),
            (     'gt', '4', 'xywh'),
            (   'kbrs', '0', 'xywh'),
            (   'kbrs', '1', 'xywh'),
            (   'kbrs', '2', 'xywh'),
            (   'kbrs', '3', 'xywh'),
            (   'kbrs', '4', 'xywh'),
            ('vanilla', '0', 'xywh'),
            ('vanilla', '1', 'xywh'),
            ('vanilla', '2', 'xywh'),
            ('vanilla', '3', 'xywh'),
            ('vanilla', '4', 'xywh')],
           names=['group', 'id', 'metric'])

In [168]:
import numpy as np
import pandas as pd
import ast, math, re
idx = pd.IndexSlice

# ---------- robust (x,y) 파서 ----------
def parse_xy(v):
    if v is None: return np.nan, np.nan
    if isinstance(v, str):
        s = v.strip()
        if s == "" or s.lower() in {"nan","none","null"}:
            return np.nan, np.nan
        try:
            v = ast.literal_eval(s)  # "[108,2,20,12]" → [108,2,20,12]
        except Exception:
            m = re.search(r'x\s*[:=]\s*([-+]?\d*\.?\d+)', s)
            n = re.search(r'y\s*[:=]\s*([-+]?\d*\.?\d+)', s)
            if m and n:
                return float(m.group(1)), float(n.group(1))
            return np.nan, np.nan
    if isinstance(v, (list, tuple, np.ndarray)):
        if len(v) >= 2:
            try: return float(v[0]), float(v[1])
            except Exception: return np.nan, np.nan
    if isinstance(v, (dict, pd.Series)):
        for kx,ky in (('x','y'),('cx','cy'),(0,1),('X','Y')):
            try: return float(v[kx]), float(v[ky])
            except Exception: pass
        return np.nan, np.nan
    try:
        if math.isnan(v): return np.nan, np.nan
    except Exception:
        pass
    return np.nan, np.nan

def xy_block(df: pd.DataFrame, group: str) -> np.ndarray:
    sub = df.loc[:, idx[group, :, 'xywh']]
    N, M = len(df), sub.shape[1]
    out = np.full((N, M, 2), np.nan, dtype=float)
    for j, col in enumerate(sub.columns):
        out[:, j, :] = np.array([parse_xy(v) for v in sub[col].to_numpy()], dtype=float)
    return out  # (N,M,2)

# ---------- pairwise 통계 ----------
def pairwise_row_stats(A: np.ndarray, B: np.ndarray, index, prefix: str):
    """
    A:(N,a,2), B:(N,b,2) → 각 row의 모든 (a×b) 유클리드 거리 통계
    반환: (row_stats_df, overall_summary_dict)
    """
    N = A.shape[0]
    # (N,a,b,2) → (N,a,b)
    diff = A[:, :, None, :] - B[:, None, :, :]
    d = np.sqrt(np.sum(diff**2, axis=-1))
    # 유효 마스크
    maskA = ~np.isnan(A).any(-1)       # (N,a)
    maskB = ~np.isnan(B).any(-1)       # (N,b)
    mask  = maskA[:, :, None] & maskB[:, None, :]  # (N,a,b)

    valid_counts = mask.sum(axis=(1,2))               # (N,)
    d_masked = np.where(mask, d, np.nan)              # (N,a,b)
    d_flat   = d_masked.reshape(N, -1)                # (N, a*b)

    # row별 sum / sumsq (빈 슬라이스 경고 회피)
    row_sum  = np.nansum(d_flat, axis=1)
    row_sum2 = np.nansum(d_flat**2, axis=1)

    row_mean = np.where(valid_counts>0, row_sum / valid_counts, np.nan)
    # population std (원하면 샘플표준편차로 바꾸기)
    row_var  = np.where(valid_counts>0, row_sum2/valid_counts - row_mean**2, np.nan)
    row_std  = np.sqrt(np.maximum(row_var, 0.0))

    # min/max/percentiles
    with np.errstate(all='ignore'):
        row_min = np.nanmin(d_flat, axis=1)
        row_max = np.nanmax(d_flat, axis=1)
        q25, q50, q75 = np.nanpercentile(d_flat, [25,50,75], axis=1)

    stats_df = pd.DataFrame({
        f'{prefix}_mean'  : row_mean,
        f'{prefix}_std'   : row_std,
        f'{prefix}_min'   : row_min,
        f'{prefix}_p25'   : q25,
        f'{prefix}_median': q50,
        f'{prefix}_p75'   : q75,
        f'{prefix}_max'   : row_max,
        f'{prefix}_valid_pairs': valid_counts.astype(float),
    }, index=index)

    # ----- 전체(가중) 통계 -----
    total_pairs = int(np.nansum(valid_counts))
    overall_mean_weighted = float(np.nansum(row_sum) / total_pairs) if total_pairs>0 else float('nan')
    overall_var_weighted  = float(np.nansum(row_sum2) / total_pairs - overall_mean_weighted**2) if total_pairs>0 else float('nan')
    overall_std_weighted  = float(np.sqrt(max(overall_var_weighted, 0.0))) if total_pairs>0 else float('nan')

    # 전체 min/max/중앙값(모든 유효 pair 기준)
    # 메모리 여유 있으면 다음 한 줄로 전체 값 펼쳐서 계산 가능
    flat_all = d_flat[~np.isnan(d_flat)]
    overall_min   = float(np.min(flat_all))  if flat_all.size else float('nan')
    overall_max   = float(np.max(flat_all))  if flat_all.size else float('nan')
    overall_med   = float(np.median(flat_all)) if flat_all.size else float('nan')
    overall_p25   = float(np.percentile(flat_all, 25)) if flat_all.size else float('nan')
    overall_p75   = float(np.percentile(flat_all, 75)) if flat_all.size else float('nan')

    summary = {
        'rows_with_any_pairs': int((valid_counts>0).sum()),
        'total_valid_pairs': total_pairs,
        'overall_mean_weighted': overall_mean_weighted,
        'overall_std_weighted' : overall_std_weighted,
        'overall_min'          : overall_min,
        'overall_p25'          : overall_p25,
        'overall_median'       : overall_med,
        'overall_p75'          : overall_p75,
        'overall_max'          : overall_max,
        # 참고용: 행평균의 평균(가중치 없이)
        'mean_of_row_means'    : float(np.nanmean(row_mean)) if np.any(valid_counts>0) else float('nan'),
    }
    return stats_df, summary

# ---------- 사용 ----------
df = tmp

gt_xy      = xy_block(df, 'gt')
kbrs_xy    = xy_block(df, 'kbrs')
vanilla_xy = xy_block(df, 'vanilla')

# gt vs kbrs
stats_kbrs, summary_kbrs = pairwise_row_stats(gt_xy, kbrs_xy, df.index, prefix='pair_gt_kbrs')
# gt vs vanilla
stats_van , summary_van  = pairwise_row_stats(gt_xy, vanilla_xy, df.index, prefix='pair_gt_van')

In [169]:
# df.columns.names 가 ['group','id','metric'] 라고 가정
col_names = list(df.columns.names)  # ['group','id','metric']

def lift_pair_stats(stats_df: pd.DataFrame, id_label: str, prefix: str) -> pd.DataFrame:
    """
    stats_df: columns = ['pair_gt_kbrs_mean', 'pair_gt_kbrs_std', ...] 같은 1-레벨
    id_label: 'gt_kbrs' / 'gt_van' 등 (pair 안쪽 서브그룹 이름)
    prefix  : 'pair_gt_kbrs_' / 'pair_gt_van_' -> metric 이름에서 제거
    반환: 3-레벨 MultiIndex columns로 변환된 DF
          top-level='pair', second-level=id_label, third-level=metric (mean, std, ...)
    """
    s = stats_df.rename(columns=lambda c: str(c).replace(prefix, ''), copy=True)
    tuples = [('pair', id_label, c) for c in s.columns]
    # s.columns = pd.MultiIndex.from_tuples(tuples)
    return s

# 통계 DF를 3-레벨로 승격
stats_kbrs_l = lift_pair_stats(stats_kbrs, id_label='gt_kbrs', prefix='pair_gt_kbrs_')
stats_van_l  = lift_pair_stats(stats_van,  id_label='gt_van',  prefix='pair_gt_van_')

# join 대신 concat으로 한 번에 병합 (index 기준 정렬/정합)
# df = pd.concat([df, stats_kbrs_l, stats_van_l], axis=1)

In [170]:
pd.DataFrame({
        'gt_kbrs': stats_kbrs_l['mean'],
        'gt_vanilla': stats_van_l['mean'],
}
).describe()

,gt_kbrs,gt_vanilla
count,148659.000000,148659.000000
mean,54.573715,57.896541
std,19.829763,17.385519
min,4.137248,4.276373
25%,40.325215,46.228739
50%,57.801279,60.692752
75%,70.358565,71.002831
max,108.255367,126.528869
